# C++ Tracker 評測 (mAP / HOTA / MOTA / IDF1)

流程：

1. 由 `val.json` 產生 `image_manifest.tsv`（依 `video_id`、`frame_index` 排序）
2. 跑 C++ 執行檔 → `predict.txt`（MOT 格式）、`det_coco.json`、`track_coco.json`
3. 依 `video_id` 把 GT 與預測拆成多個 sequence，frame 重新編號成 1..N
4. TrackEval 逐類別算 HOTA / MOTA / IDF1
5. pycocotools 算 mAP

**重要**：C++ 端 `predict.txt` 的第 1 欄請填 `val.json` 的 `image_id`，不要用自己累加的 frame 編號。

In [1]:
# ===== 路徑設定 =====
GT_JSON      = "/media/jianhua/HDD 1T/dataset/backup/val.json"
IMAGE_FOLDER = "/media/jianhua/HDD 1T/dataset/SeaDronesSee_MOT/images/val"
IMG_EXT      = ".jpg"                 # val.json 裡是 .png，實際檔案若是 jpg 就填 .jpg

CPP_BIN   = "build/eval"      # 你的 C++ 執行檔
ONNX_PATH = "/home/jianhua/Desktop/vitis-ai-pytorch/dpu_inference/model/YOLO_int.onnx"     # ONNX 模型
CONF_TH   = 0.001                       # 和 C++ 端的預設值保持一致
IOU_TH    = 0.5
SEQ_FPS   = 30                        # 換算 t_cap 用，要和原影片幀率一致
IO_DIR    = "trackeval_io"            # C++ 輸出資料夾
WORK_DIR  = "trackeval_tmp"           # TrackEval 暫存資料夾
MANIFEST  = f"{IO_DIR}/image_manifest.tsv"
PRED_TXT  = f"{IO_DIR}/predict.txt"
DET_JSON  = f"{IO_DIR}/det_coco.json"
TRK_JSON  = f"{IO_DIR}/track_coco.json"

DATA_YAML = "/home/jianhua/Desktop/vitis-ai-pytorch/train/config/data.yaml"
# 模型 class_id -> val.json category_id 的對應表由下一格從 data.yaml 自動產生

import os
os.makedirs(IO_DIR, exist_ok=True)

## 1. 由 `data.yaml` 產生 class 對應表

用類別名稱把 `data.yaml` 的 `names`（模型 class_id）對到 `val.json` 的 `category_id`，
避免手動填錯順序。名稱比對會忽略大小寫、底線與多餘空白；
對不上的用 `NAME_ALIAS` 補（例如 `lifejacket` → `life jacket`）。

結果會組成 `CLS_MAP_ARG`，下一步直接用 `--cls-map` 傳給 C++，
所以換資料集或改 `data.yaml` 都不必動 C++ 原始碼、也不用重編。

In [2]:
import json, os, re

# 名稱對不上時在這裡補；key = data.yaml 的名稱，value = val.json 的名稱
NAME_ALIAS = {
    "swimmer with lifejacket": "swimmer with life jacket",
    "lifejacket":              "life jacket",
    # "floater":               "swimmer",
}


def _norm(s):
    s = re.sub(r"[_\-]+", " ", str(s).strip().lower())
    return re.sub(r"\s+", " ", s)


def _parse_names_fallback(path):
    # 沒裝 pyyaml 時的簡易 names 解析
    txt = open(path, "r", encoding="utf-8").read()
    m = re.search(r"^names\s*:\s*\[(.*?)\]", txt, re.S | re.M)
    if m:
        return [x.strip().strip("'\"") for x in m.group(1).split(",") if x.strip()]
    m = re.search(r"^names\s*:\s*$(.*?)(?=^\S|\Z)", txt, re.S | re.M)
    if m:
        out = {}
        for line in m.group(1).splitlines():
            mm = re.match(r"\s*(\d+)\s*:\s*(.+?)\s*$", line)
            if mm:
                out[int(mm.group(1))] = mm.group(2).strip().strip("'\"")
                continue
            mm = re.match(r"\s*-\s*(.+?)\s*$", line)
            if mm:
                out[len(out)] = mm.group(1).strip().strip("'\"")
        return out
    return None


def load_yaml_names(path):
    try:
        import yaml
        with open(path, "r", encoding="utf-8") as f:
            names = (yaml.safe_load(f) or {}).get("names")
    except ImportError:
        names = _parse_names_fallback(path)
    if names is None:
        raise ValueError(f"{path} 裡找不到 names")
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    return dict(enumerate(names))


def build_cls_map(data_yaml, gt_json, alias=None):
    alias = {_norm(k): _norm(v) for k, v in {**NAME_ALIAS, **(alias or {})}.items()}
    names = load_yaml_names(data_yaml)
    with open(gt_json, "r", encoding="utf-8") as f:
        cats = json.load(f)["categories"]
    gt_by_name = {_norm(c["name"]): c["id"] for c in cats}

    mapping, unmatched = {}, []
    for cid in sorted(names):
        n = _norm(names[cid])
        n = alias.get(n, n)
        if n in gt_by_name:
            mapping[cid] = gt_by_name[n]
        else:
            unmatched.append((cid, names[cid]))
    return mapping, names, gt_by_name, unmatched


PRED_CLS_TO_COCO_CAT, YAML_NAMES, GT_BY_NAME, UNMATCHED = build_cls_map(
    DATA_YAML, GT_JSON)

print(f"data.yaml names ({len(YAML_NAMES)} 類):")
for cid in sorted(YAML_NAMES):
    cat = PRED_CLS_TO_COCO_CAT.get(cid)
    mark = f"-> category_id {cat}" if cat is not None else "-> ❌ 對不到"
    print(f"  cls {cid}: {YAML_NAMES[cid]:<28} {mark}")

unused = set(GT_BY_NAME.values()) - set(PRED_CLS_TO_COCO_CAT.values())
if unused:
    print(f"\n⚠️ val.json 有類別沒被模型覆蓋: {sorted(unused)}")
if UNMATCHED:
    print(f"\n❌ 這些名稱對不到 val.json，請在 NAME_ALIAS 補上: {UNMATCHED}")
    print(f"   val.json 可用名稱: {sorted(GT_BY_NAME)}")

# -1 = 該 class 在 val.json 沒有對應類別，寫檔時會被跳過（保留索引位置）
CLS_MAP_LIST = [PRED_CLS_TO_COCO_CAT.get(k, -1) for k in range(max(YAML_NAMES) + 1)]
CLS_MAP_ARG  = ",".join(map(str, CLS_MAP_LIST))   # 下一步用 --cls-map 傳給 C++
print(f"\n--cls-map {CLS_MAP_ARG}")

data.yaml names (4 類):
  cls 0: swimmer                      -> category_id 1
  cls 1: swimmer_with_life_jacket     -> category_id 2
  cls 2: boat                         -> category_id 3
  cls 3: life_jacket                  -> category_id 6

--cls-map 1,2,3,6


## 2. 產生 image manifest（給 C++ 讀）

In [3]:
import json, os
from collections import defaultdict

def build_manifest(gt_json, image_folder, out_tsv, img_ext=".jpg", check_exists=True):
    """依 (video_id, frame_index) 排序輸出：image_id \t video_id \t frame_index \t path"""
    with open(gt_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows, missing = [], []
    for img in sorted(data["images"], key=lambda x: (x["video_id"], x["frame_index"])):
        name = os.path.splitext(img["file_name"])[0] + img_ext
        path = os.path.join(image_folder, name)
        if check_exists and not os.path.exists(path):
            missing.append(path)
            continue
        rows.append(f"{img['id']}\t{img['video_id']}\t{img['frame_index']}\t{path}")

    os.makedirs(os.path.dirname(out_tsv) or ".", exist_ok=True)
    with open(out_tsv, "w", encoding="utf-8") as f:
        f.write("\n".join(rows) + "\n")
    return rows, missing

rows, missing = build_manifest(GT_JSON, IMAGE_FOLDER, MANIFEST, IMG_EXT)
print(f"✅ manifest: {len(rows)} 張影像 -> {MANIFEST}")
if missing:
    print(f"⚠️ 有 {len(missing)} 張圖找不到，例如: {missing[:3]}")
print("\n".join(rows[:3]))

✅ manifest: 8584 張影像 -> trackeval_io/image_manifest.tsv
267	0	572	/media/jianhua/HDD 1T/dataset/SeaDronesSee_MOT/images/val/267.jpg
268	0	573	/media/jianhua/HDD 1T/dataset/SeaDronesSee_MOT/images/val/268.jpg
269	0	574	/media/jianhua/HDD 1T/dataset/SeaDronesSee_MOT/images/val/269.jpg


## 3. 執行 C++ 追蹤

C++ 端會輸出三個檔：

| 檔案 | 用途 |
|---|---|
| `predict.txt` | MOT 格式，`image_id,track_id,x,y,w,h,score,class_id,1` |
| `det_coco.json` | 追蹤前的偵測結果，算「偵測 mAP」 |
| `track_coco.json` | 追蹤後的框，算「追蹤 mAP」 |

> `--conf` 沿用實機的門檻（0.2），量到的就是實際部署設定下的表現。
> 若要跟 ultralytics 的 mAP 對齊（它預設用 conf=0.001），要另外跑一趟 `--conf 0.001`
> 再拿那次的 `det_coco.json` 算，否則低分框被切掉、mAP 會被低估。

> 這一格會即時解析 C++ 的 `PROGRESS` 輸出畫進度條（有裝 tqdm 就用 tqdm，沒有就退回單行覆寫）。


In [4]:
import os, re, shlex, subprocess, time, codecs
from collections import deque

CMD = (f"{CPP_BIN} {shlex.quote(ONNX_PATH)}"
       f" --manifest {shlex.quote(MANIFEST)}"
       f" --out-dir {shlex.quote(IO_DIR)}"
       f" --conf {CONF_TH} --iou {IOU_TH} --fps {SEQ_FPS}"
       f" --cls-map {CLS_MAP_ARG}")
print("執行:", CMD, "\n")

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

_PROG = re.compile(r"PROGRESS (\d+)/(\d+)"
                   r"(?: det=(\d+))?(?: trk=(\d+))?(?: fps=([\d.]+))?")


def run_with_progress(cmd, total=None, log_lines=12):
    # 即時解析 C++ 的 PROGRESS 輸出畫進度條，回傳 (returncode, 最後幾行 log)
    proc = subprocess.Popen(shlex.split(cmd),
                            stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT)

    bar = tqdm(total=total, unit="frame", dynamic_ncols=True) if tqdm else None
    dec = codecs.getincrementaldecoder("utf-8")("replace")   # 中文可能被切在 chunk 邊界
    tail, buf, t0 = deque(maxlen=log_lines), "", time.time()

    def emit(line):
        line = line.strip()
        if not line:
            return
        m = _PROG.search(line)
        if m:
            cur, tot = int(m.group(1)), int(m.group(2))
            if bar is None:
                print("\r" + line, end="", flush=True)
                return
            if bar.total != tot:
                bar.total = tot
                bar.refresh()
            bar.update(cur - bar.n)
            bar.set_postfix_str(f"det={m.group(3) or '-'} trk={m.group(4) or '-'} "
                                f"fps={m.group(5) or '-'}")
        else:
            tail.append(line)
            bar.write(line) if bar is not None else print(line, flush=True)

    try:
        while True:
            chunk = proc.stdout.read1(4096)
            if not chunk:
                break
            buf += dec.decode(chunk)
            parts = re.split(r"[\r\n]", buf)
            buf = parts.pop()          # 最後一段可能還沒收完，留到下一輪
            for part in parts:
                emit(part)
        if buf:
            emit(buf)
    except KeyboardInterrupt:
        proc.terminate()
        raise
    finally:
        proc.stdout.close()
        proc.wait()
        if bar is not None:
            bar.close()

    print(f"\n⏱️ 總耗時 {time.time() - t0:.1f}s  (returncode={proc.returncode})")
    return proc.returncode, list(tail)


rc, tail = run_with_progress(CMD, total=len(rows))

if rc != 0:
    print("❌ C++ 執行失敗，最後幾行輸出：")
    for ln in tail:
        print("  ", ln)
else:
    for p in (PRED_TXT, DET_JSON, TRK_JSON):
        size = f"{os.path.getsize(p)/1e6:.1f} MB" if os.path.exists(p) else "缺少"
        print(f"  {p}: {size}")

/home/jianhua/miniconda3/envs/mot-eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


執行: build/eval /home/jianhua/Desktop/vitis-ai-pytorch/dpu_inference/model/YOLO_int.onnx --manifest trackeval_io/image_manifest.tsv --out-dir trackeval_io --conf 0.001 --iou 0.5 --fps 30 --cls-map 1,2,3,6 



  0%|          | 0/8584 [00:00<?, ?frame/s]

[evalio] manifest: 8584 images
模型輸入: 640x640  輸出數量: 3
推導出的 nc = 4
class 對照表 (模型 cls -> category_id): 0->1, 1->2, 2->3, 3->6
[video 0] start


  2%|▏         | 140/8584 [00:10<09:05, 15.48frame/s, det=10 trk=3 fps=13.81]

...

[video 19] start


 82%|████████▏ | 7010/8584 [07:26<01:31, 17.15frame/s, det=4 trk=1 fps=15.70]

[video 21] start


100%|██████████| 8584/8584 [08:57<00:00, 15.97frame/s, det=20 trk=9 fps=15.99] 

[evalio] wrote 35039 rows -> trackeval_io/predict.txt
[evalio] wrote 35039 rows -> trackeval_io/track_coco.json
[evalio] wrote 116993 rows -> trackeval_io/det_coco.json
共處理 8584 幀，track rows = 35039，det rows = 116993

⏱️ 總耗時 537.5s  (returncode=0)
  trackeval_io/predict.txt: 1.7 MB
  trackeval_io/det_coco.json: 12.1 MB
  trackeval_io/track_coco.json: 3.6 MB


## 4. 轉成 TrackEval 格式

- 每支影片一個 sequence（`video_00`, `video_01`, ...），frame 重新編號成 1..N
- 每個 category 各建一份資料夾，GT 與預測的 class 都改標成 `pedestrian=1`（TrackEval 預設只評這類）
- 同幀重複 id 會自動剔除

In [5]:
import json, os, shutil
from collections import defaultdict

def build_frame_index(data):
    """image_id -> (seq_name, local_frame 1-based)，另回傳每個 seq 的長度"""
    by_video = defaultdict(list)
    for img in data["images"]:
        by_video[img["video_id"]].append(img)

    img2frame, seq_len = {}, {}
    for vid, imgs in by_video.items():
        imgs.sort(key=lambda x: x["frame_index"])
        seq = f"video_{vid:02d}"
        for i, img in enumerate(imgs, start=1):
            img2frame[img["id"]] = (seq, i)
        seq_len[seq] = len(imgs)
    return img2frame, seq_len


def _dedup(rows):
    seen, out, dropped = set(), [], 0
    for r in rows:
        if (r[0], r[1]) in seen:
            dropped += 1
            continue
        seen.add((r[0], r[1]))
        out.append(r)
    return out, dropped


def _write_rows(path, rows):
    """每行自己帶換行；沒資料就寫成 0 byte 的空檔。

    ★ TrackEval 用 `if fp.tell():` 判斷檔案是否為空，所以「空檔」必須真的是
      0 byte。若寫成單一個 "\n"，它會拿空行去 csv.Sniffer().sniff()，
      丟出 "Could not determine delimiter" → "invalidly formatted"。
    """
    with open(path, "w") as f:
        for r in rows:
            f.write(",".join(str(v) for v in r) + "\n")


def _write_seqinfo(seq_dir, seq_name, length, w=3840, h=2160):
    with open(os.path.join(seq_dir, "seqinfo.ini"), "w") as f:
        f.write("[Sequence]\n"
                f"name={seq_name}\nimDir=img1\nframeRate=30\n"
                f"seqLength={length}\nimWidth={w}\nimHeight={h}\nimExt=.jpg\n")


def prepare_trackeval(gt_json, pred_txt, work_dir, pred_cls_to_coco_cat,
                      tracker_name="cpp_tracker"):
    with open(gt_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    cat_name  = {c["id"]: c["name"] for c in data["categories"]}
    img2frame, seq_len = build_frame_index(data)
    W = data["videos"][0].get("width", 3840)
    H = data["videos"][0].get("height", 2160)

    # GT
    gt_rows = defaultdict(list)
    for ann in data["annotations"]:
        loc = img2frame.get(ann["image_id"])
        if loc is None:
            continue
        seq, fr = loc
        x, y, w, h = ann["bbox"]
        gt_rows[(ann["category_id"], seq)].append(
            (fr, ann["track_id"], float(x), float(y), float(w), float(h), 1, 1, 1))

    # 預測
    pred_rows = defaultdict(list)
    unknown_img = unknown_cls = 0
    with open(pred_txt, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p = line.split(",")
            image_id, tid = int(p[0]), int(p[1])
            x, y, w, h = (float(v) for v in p[2:6])
            score, cls = float(p[6]), int(p[7])

            cat = pred_cls_to_coco_cat.get(cls)
            if cat is None:
                unknown_cls += 1
                continue
            loc = img2frame.get(image_id)
            if loc is None:
                unknown_img += 1
                continue
            seq, fr = loc
            pred_rows[(cat, seq)].append((fr, tid, x, y, w, h, score, 1, 1))

    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)

    layout, empty_pred = {}, []
    for cat in sorted({c for c, _ in gt_rows}):
        root, seqs = os.path.join(work_dir, f"cat_{cat}"), {}
        for (c, seq), rws in gt_rows.items():
            if c != cat:
                continue
            gt_dir = os.path.join(root, "gt", seq, "gt")
            os.makedirs(gt_dir, exist_ok=True)
            g, _ = _dedup(sorted(rws))
            _write_rows(os.path.join(gt_dir, "gt.txt"), g)
            _write_seqinfo(os.path.join(root, "gt", seq), seq, seq_len[seq], W, H)

            tr_dir = os.path.join(root, "trackers", tracker_name, "data")
            os.makedirs(tr_dir, exist_ok=True)
            p, _ = _dedup(sorted(pred_rows.get((cat, seq), [])))
            _write_rows(os.path.join(tr_dir, f"{seq}.txt"), p)
            if not p:
                empty_pred.append((cat, seq))
            seqs[seq] = seq_len[seq]

        layout[cat] = {"name": cat_name.get(cat, str(cat)), "root": root, "seqs": seqs}

    return layout, {"unknown_image_id": unknown_img,
                    "unknown_class": unknown_cls,
                    "empty_pred": empty_pred}

In [6]:
TRACKER_NAME = "cpp_tracker"
layout, stats = prepare_trackeval(GT_JSON, PRED_TXT, WORK_DIR,
                                  PRED_CLS_TO_COCO_CAT, TRACKER_NAME)

if stats["unknown_image_id"]:
    print(f"⚠️ predict.txt 有 {stats['unknown_image_id']} 筆 image_id 不在 GT 裡"
          f"（第一欄是不是填成累加的 frame number 了？）")
if stats["unknown_class"]:
    print(f"⚠️ 有 {stats['unknown_class']} 筆 class_id 不在 PRED_CLS_TO_COCO_CAT 裡")

if stats["empty_pred"]:
    print(f"ℹ️ 有 {len(stats['empty_pred'])} 個 (類別, 影片) 完全沒有預測框，"
          f"這些序列的 GT 會全部算成 FN（正常現象，不是錯誤）：")
    for cat, seq in stats["empty_pred"][:10]:
        print(f"     cat {cat} / {seq}")

for cat, info in layout.items():
    print(f"cat {cat:>2} [{info['name']}] : {len(info['seqs'])} 個 sequence, "
          f"共 {sum(info['seqs'].values())} 幀")

ℹ️ 有 8 個 (類別, 影片) 完全沒有預測框，這些序列的 GT 會全部算成 FN（正常現象，不是錯誤）：
     cat 1 / video_00
     cat 1 / video_01
     cat 1 / video_17
     cat 2 / video_01
     cat 2 / video_02
     cat 2 / video_06
     cat 2 / video_10
     cat 2 / video_13
cat  1 [swimmer] : 10 個 sequence, 共 6545 幀
cat  2 [swimmer with life jacket] : 13 個 sequence, 共 7788 幀
cat  3 [boat] : 16 個 sequence, 共 8441 幀


## 5. HOTA / MOTA / IDF1

In [7]:
# ── numpy / TrackEval 相容性補丁 ────────────────────────────────────────
# numpy >= 1.24 移除了 np.float / np.int / np.bool
# numpy >= 2.0  又移除了 np.Inf / np.NaN / np.asfarray ...
# TrackEval 和舊版 scipy 都有 `from numpy import Inf, asfarray` 這種寫法，
# 所以補丁一定要在 import trackeval 之前跑。
import numpy as np

_NP_ALIAS = {
    # 常數 / 型別
    "float": float, "int": int, "bool": bool, "complex": complex,
    "Inf": np.inf, "Infinity": np.inf, "infty": np.inf, "NINF": -np.inf,
    "PINF": np.inf, "NaN": np.nan, "NAN": np.nan,
    "float_": np.float64, "int_": np.int64, "bool_": np.bool_,
    "complex_": np.complex128, "unicode_": np.str_, "longfloat": np.longdouble,
    # 函式
    "asfarray": lambda a, dtype=np.float64: np.asarray(a, dtype=dtype),
    "alltrue": np.all, "sometrue": np.any, "product": np.prod,
    "cumproduct": np.cumprod, "round_": np.round,
}
for _n, _v in _NP_ALIAS.items():
    if not hasattr(np, _n):
        setattr(np, _n, _v)

try:
    import trackeval
    print(f"numpy {np.__version__} / trackeval OK")
except ImportError as _e:
    import scipy
    print("❌ import trackeval 失敗:", _e)
    print(f"   numpy {np.__version__} / scipy {scipy.__version__}")
    print("   這是 numpy 2.x 搭配 numpy 1.x 時代的 scipy / pycocotools 造成的。")
    print("   最省事的解法（裝完要重啟 kernel）：")
    print('     pip install "numpy==1.26.4"')
    raise


def eval_tracking(info, tracker_name=TRACKER_NAME):
    eval_cfg = trackeval.Evaluator.get_default_eval_config()
    eval_cfg.update({"PRINT_CONFIG": False, "PRINT_RESULTS": False,
                     "DISPLAY_LESS_PROGRESS": True, "TIME_PROGRESS": False,
                     "OUTPUT_SUMMARY": False, "OUTPUT_DETAILED": False,
                     "PLOT_CURVES": False, "USE_PARALLEL": False})

    ds_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    ds_cfg.update({"GT_FOLDER": os.path.join(info["root"], "gt"),
                   "TRACKERS_FOLDER": os.path.join(info["root"], "trackers"),
                   "TRACKERS_TO_EVAL": [tracker_name],
                   "SKIP_SPLIT_FOL": True,
                   "DO_PREPROC": False,          # 我們不是 MOT17，不要跑 distractor 前處理
                   "PRINT_CONFIG": False,
                   "SEQ_INFO": info["seqs"],
                   "CLASSES_TO_EVAL": ["pedestrian"]})

    metrics = [trackeval.metrics.HOTA({"PRINT_CONFIG": False}),
               trackeval.metrics.CLEAR({"PRINT_CONFIG": False}),
               trackeval.metrics.Identity({"PRINT_CONFIG": False})]

    evaluator = trackeval.Evaluator(eval_cfg)
    dataset   = trackeval.datasets.MotChallenge2DBox(ds_cfg)
    results, _ = evaluator.evaluate([dataset], metrics)
    return results["MotChallenge2DBox"][tracker_name]["COMBINED_SEQ"]["pedestrian"]


def _m(v):
    return float(v.mean()) if hasattr(v, "mean") else float(v)

summary = {}
for cat, info in layout.items():
    print(f"\n{'='*50}\n🚀 評估類別: {info['name']} (cat {cat})\n{'='*50}")
    r = eval_tracking(info)
    summary[info["name"]] = {
        "HOTA":  _m(r["HOTA"]["HOTA"]) * 100,
        "DetA":  _m(r["HOTA"]["DetA"]) * 100,
        "AssA":  _m(r["HOTA"]["AssA"]) * 100,
        "MOTA":  _m(r["CLEAR"]["MOTA"]) * 100,
        "MOTP":  _m(r["CLEAR"]["MOTP"]) * 100,
        "IDF1":  _m(r["Identity"]["IDF1"]) * 100,
        "IDSW":  int(r["CLEAR"]["IDSW"]),
        "FP":    int(r["CLEAR"]["CLR_FP"]),
        "FN":    int(r["CLEAR"]["CLR_FN"]),
    }

import pandas as pd
df = pd.DataFrame(summary).T
print("\n========== TRACKING METRICS ==========")
print(df.round(2).to_string())

/tmp/ipykernel_3203541/2598029984.py:21: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _n):


Error importing BURST due to missing underlying dependency: No module named 'tabulate'
numpy 1.26.4 / trackeval OK

🚀 評估類別: swimmer (cat 1)

Evaluating 1 tracker(s) on 10 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating cpp_tracker


🚀 評估類別: swimmer with life jacket (cat 2)

Evaluating 1 tracker(s) on 13 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating cpp_tracker


🚀 評估類別: boat (cat 3)

Evaluating 1 tracker(s) on 16 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating cpp_tracker


========== TRACKING METRICS ==========
                           HOTA   DetA   AssA   MOTA   MOTP   IDF1  IDSW      FP      FN
swimmer                   46.26  41.97  51.03  45.95  73.59  60.21   7.0  1088.0  4365.0
swimmer with life jacket  36.84  38.64  35.25  49.02  72.04  4

## 6. mAP (pycocotools)

In [8]:
import contextlib, io as _io
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def coco_eval(gt_json, dt_json, tag=""):
    if not os.path.exists(dt_json):
        print(f"⚠️ 找不到 {dt_json}，略過")
        return None

    with open(gt_json) as f:
        data = json.load(f)
    for a in data["annotations"]:                     # pycocotools 需要這兩個欄位
        a.setdefault("iscrowd", 0)
        a.setdefault("area", a["bbox"][2] * a["bbox"][3])
    fixed = os.path.join(IO_DIR, "gt_cocoeval.json")
    with open(fixed, "w") as f:
        json.dump(data, f)

    with contextlib.redirect_stdout(_io.StringIO()):
        coco = COCO(fixed)
        dt   = coco.loadRes(dt_json)
        e = COCOeval(coco, dt, "bbox")
        e.evaluate(); e.accumulate()

    print(f"\n{'='*50}\n📦 {tag}\n{'='*50}")
    e.summarize()

    # 每類別 AP50-95
    cat_ids = coco.getCatIds()
    names   = {c["id"]: c["name"] for c in coco.loadCats(cat_ids)}
    print("\n--- per-class AP@[.50:.95] ---")
    for i, cid in enumerate(cat_ids):
        p = e.eval["precision"][:, :, i, 0, 2]
        p = p[p > -1]
        ap = p.mean() if p.size else float("nan")
        print(f"  {names[cid]:<28} {ap:.4f}")
    return {"mAP50-95": e.stats[0], "mAP50": e.stats[1], "mAP75": e.stats[2]}

det_map   = coco_eval(GT_JSON, DET_JSON, "偵測 mAP（追蹤前，NMS 輸出）")
track_map = coco_eval(GT_JSON, TRK_JSON, "追蹤 mAP（Kalman 平滑後的框）")


📦 偵測 mAP（追蹤前，NMS 輸出）
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.348
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.746
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.277
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.029
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.266
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.457
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.191
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.433
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.433
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.047
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.358
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.564

--- per-class AP@

In [9]:
print("\n" + "*" * 46)
print("*          FINAL SUMMARY                     *")
print("*" * 46)
if det_map:
    print(f"Detection  mAP@50-95 : {det_map['mAP50-95']*100:.2f}   mAP@50 : {det_map['mAP50']*100:.2f}")
if track_map:
    print(f"Tracking   mAP@50-95 : {track_map['mAP50-95']*100:.2f}   mAP@50 : {track_map['mAP50']*100:.2f}")
print()
print(df[["HOTA", "DetA", "AssA", "MOTA", "IDF1", "IDSW"]].round(2).to_string())


**********************************************
*          FINAL SUMMARY                     *
**********************************************
Detection  mAP@50-95 : 34.79   mAP@50 : 74.60
Tracking   mAP@50-95 : 30.45   mAP@50 : 61.41

                           HOTA   DetA   AssA   MOTA   IDF1  IDSW
swimmer                   46.26  41.97  51.03  45.95  60.21   7.0
swimmer with life jacket  36.84  38.64  35.25  49.02  47.53  38.0
boat                      68.34  65.58  71.77  82.62  80.98   3.0
